# Cospectra Plotting and Spectral Corrections with `eccospectra`

This notebook covers the complete (co)spectral analysis workflow for high-frequency
eddy covariance data logged in Campbell Scientific TOA5 format.

**Workflow:**
1. Load a 30-minute TOA5 block with `read_toa5()`
2. Configure site (`SiteConfig`) and instrument (`InstrumentConfig`)
3. Despike the raw time series with `despike_dataframe()`
4. Compute cospectra, power spectra, and ogives with `process_file()`
5. Visualize normalized cospectra against the Kaimal (1972) model
6. Explore spectral transfer functions (low- and high-frequency loss)
7. Compute spectral correction factors (Massman 2000; Horst 1997)
8. Apply corrections and compare before vs. after
9. Apply the Webb-Pearman-Leuning (WPL) density correction

**Data:** Synthetic 30-minute block at 20 Hz from a Campbell Scientific IRGASON  
(36 000 records; 30 artificial spikes injected for testing).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import copy
from pathlib import Path

import eccospectra as ec
from eccospectra.io import read_toa5
from eccospectra.core import SiteConfig, process_file
from eccospectra.corrections import (
    InstrumentConfig,
    despike_dataframe,
    enrich_results_with_means,
    apply_spectral_corrections,
    compute_spectral_correction_factor,
    combined_transfer_function,
    tf_block_average,
    tf_first_order_response,
    tf_sonic_line_averaging,
    tf_scalar_path_averaging,
    tf_sensor_separation,
    wpl_correction,
    horst_analytical_correction,
    _kaimal_cospec_model,
)
from eccospectra.plotting import plot_cospectra, plot_spectra, plot_ogive
from eccospectra.qc import fit_inertial_slope

plt.rcParams['figure.dpi'] = 110
print(f'eccospectra {ec.__version__}')

## 1. Load TOA5 Data

`read_toa5()` parses the 4-line Campbell Scientific header and returns a **Polars DataFrame**
plus a metadata dictionary with station info and column units.

The example file contains a synthetic 30-minute block at 20 Hz with 30 injected spikes.  
See `despike_demo.ipynb` for a detailed despiking walkthrough.

In [ ]:
DATA_FILE = Path('data/TOA5_ExampleSite_2024_07_15_1200.dat')

df_raw, metadata = read_toa5(DATA_FILE)

print(f'Records:    {len(df_raw):,}')
print(f'Columns:    {df_raw.columns}')
ts0, ts1 = df_raw['TIMESTAMP'][0], df_raw['TIMESTAMP'][-1]
print(f'Time range: {ts0}  ->  {ts1}')

station = metadata['station_id']
logger  = metadata['logger_model']
print(f'\nStation: {station}')
print(f'Logger:  {logger}')
print('\nUnits:')
for col, unit in metadata['units'].items():
    print(f'  {col:<20s} {unit}')

## 2. Configure Site and Instrument

`SiteConfig` defines tower geometry and processing parameters. Displacement height
$d = (2/3)\,h_c$ and roughness length $z_0 = 0.1\,h_c$ are estimated from the canopy height
if not supplied explicitly.

`InstrumentConfig` stores the physical sensor properties needed to build spectral
transfer functions. The defaults correspond to a **Campbell Scientific IRGASON**
(integrated open-path sonic anemometer + CO$_2$/H$_2$O gas analyser).

| Parameter | IRGASON default | Effect |
|-----------|-----------------|--------|
| `sonic_path_length` | 0.10 m | Path-averaging TF |
| `irga_path_length`  | 0.154 m | Scalar path-averaging TF |
| `tau_co2` / `tau_h2o` | 0.1 s | First-order response TF |
| `sensor_separation_total` | 0 m | Co-location TF (ideal) |

In [ ]:
config = SiteConfig(
    z_measurement=3.0,    # EC tower height [m]
    z_canopy=0.3,         # canopy height [m]
    sampling_freq=20.0,   # Hz
    averaging_period=30.0 # minutes
)

print('--- Site configuration ---')
print(f'  z (measurement):  {config.z_measurement:.1f} m')
print(f'  d (displacement): {config.d:.2f} m  (= 2/3 * z_canopy)')
print(f'  z0 (roughness):   {config.z0:.3f} m  (= 0.1 * z_canopy)')
print(f'  z_eff = z - d:    {config.z_eff:.2f} m')
print(f'  Sampling freq:    {config.sampling_freq} Hz')
print(f'  Averaging period: {config.averaging_period} min')

instrument = InstrumentConfig()  # defaults: Campbell Scientific IRGASON

print('\n--- Instrument configuration ---')
print(f'  Model:              {instrument.model}')
print(f'  IRGA type:          {instrument.irga_type}')
print(f'  Sonic path length:  {instrument.sonic_path_length} m')
print(f'  IRGA path length:   {instrument.irga_path_length} m')
print(f'  tau_co2:            {instrument.tau_co2} s')
print(f'  tau_h2o:            {instrument.tau_h2o} s')
print(f'  tau_T (sonic):      {instrument.tau_T} s  (instantaneous)')
print(f'  Sensor separation:  {instrument.sensor_separation_total:.3f} m  (IRGASON: integrated)')

## 3. Despike the Raw Data

Spikes and outliers contaminate cospectral estimates at high frequencies and must be
removed before FFT computation. `despike_dataframe()` applies the iterative UKDE method
(Metzger et al. 2012) to each sensor channel.

| Parameter | Default | Effect |
|-----------|---------|--------|
| `prob_threshold` | 1e-4 | Lower = more permissive; higher = more aggressive |
| `max_iter` | 10 | Max passes (usually converges in 2-4 iterations) |

> **See also:** `despike_demo.ipynb` for a detailed algorithm walkthrough with
> before/after visualizations and sensitivity to `prob_threshold`.

In [ ]:
ec_cols = ['Ux', 'Uy', 'Uz', 'T_SONIC', 'CO2_density', 'H2O_density']

df_clean = despike_dataframe(
    df_raw,
    columns=ec_cols,
    prob_threshold=1e-4,
    max_iter=10,
    verbose=True,
)
print(f'\nShape unchanged: {df_raw.shape}  ->  {df_clean.shape}')

## 4. Compute Cospectra with `process_file()`

`process_file()` divides the DataFrame into averaging windows and for each window:
1. Screens for missing data (> 5% NaN => skip)
2. Applies double rotation (align $x$ with mean wind; set $\bar{w} = 0$)
3. Linearly detrends all six signals
4. Computes FFT-based cospectra and power spectra (Hamming window)
5. Logarithmically bins spectral estimates (20 bins/decade by default)
6. Normalizes to the area-preserving form: $n\,Co(n)\,/\,\mathrm{cov}(w', x')$
7. Computes turbulence statistics ($u_*$, Obukhov length $L$, $z/L$, $H$)
8. Integrates ogives (cumulative cospectra from high to low frequency)

Results are returned as a list of `SpectralResult` dataclass objects.

In [ ]:
results = process_file(df_clean, config, bins_per_decade=20)

print(f'Intervals processed: {len(results)}')
for i, res in enumerate(results):
    print(f'\nInterval {i + 1}: {res.timestamp_start}  ->  {res.timestamp_end}')
    print(f'  Mean wind U:     {res.u_mean:.2f} m/s   (dir: {res.wind_dir:.1f} deg)')
    print(f'  Friction vel u*: {res.ustar:.3f} m/s')
    print(f'  Stability z/L:   {res.zL:.4f}')
    print(f'  Heat flux H:     {res.H:.2f} W/m2')
    print(f'  cov(wT):         {res.cov_wT:.5f} m/s degC')
    print(f'  cov(wCO2):       {res.cov_wCO2:.5f} mg/m2/s')
    print(f'  cov(wH2O):       {res.cov_wH2O:.5f} g/m2/s')
    print(f'  Frequency bins:  {len(res.freq)}')

In [ ]:
# --- Inspect the SpectralResult for the single interval ---
res = results[0]

# Only plot where dimensionless frequency is valid (u_mean > 0.5 m/s check inside process_interval)
mask = np.isfinite(res.freq_nd) & (res.freq_nd > 0)
f_nd = res.freq_nd[mask]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'SpectralResult inspection -- z/L = {res.zL:.4f},  U = {res.u_mean:.2f} m/s', fontsize=13)

# Left: raw cospectra in physical units (semilog x)
ax = axes[0]
ax.semilogx(res.freq, res.cosp_wT, lw=1.5, label='n Co(wT)  [m/s degC]')
ax.semilogx(res.freq, res.cosp_wu, lw=1.5, label='n Co(wu)  [m2/s2]')
ax.axhline(0, color='k', lw=0.6, ls='--')
ax.set_xlabel('Natural frequency n  [Hz]', fontsize=11)
ax.set_ylabel('n * Co(n)  [physical units]', fontsize=11)
ax.set_title('Raw cospectra (physical units)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, which='both', ls=':', alpha=0.4)

# Right: normalized cospectra vs dimensionless frequency (log-log)
ax = axes[1]
for attr, lbl, color in [
    ('ncosp_wT',   'nCo(wT)/cov(wT)',     'steelblue'),
    ('ncosp_wu',   'nCo(wu)/cov(wu)',     'firebrick'),
    ('ncosp_wCO2', 'nCo(wCO2)/cov(wCO2)', 'seagreen'),
    ('ncosp_wH2O', 'nCo(wH2O)/cov(wH2O)', 'darkorchid'),
]:
    cosp = np.abs(getattr(res, attr))[mask]
    pos = cosp > 1e-5
    if pos.any():
        ax.loglog(f_nd[pos], cosp[pos], lw=1.5, color=color, label=lbl)

# Kaimal (1972) model overlays
f_model = np.logspace(-3, 2, 500)
ax.loglog(f_model, _kaimal_cospec_model(f_model, 'wT'), 'k-',  lw=2.2, alpha=0.7,
          label='Kaimal scalar')
ax.loglog(f_model, _kaimal_cospec_model(f_model, 'wu'), 'k--', lw=2.2, alpha=0.7,
          label='Kaimal momentum')

# -4/3 reference slope
f_sl = np.logspace(0.0, 1.3, 30)
ax.loglog(f_sl, 0.04 * f_sl ** (-4.0 / 3.0), color='grey', ls=':', lw=1.8, label='-4/3 slope')

ax.set_xlim(1e-2, 20)
ax.set_ylim(1e-3, 5)
ax.set_xlabel('Dimensionless frequency  f = nz/U', fontsize=11)
ax.set_ylabel('n * Co(n) / cov(wx)', fontsize=11)
ax.set_title('Normalized cospectra vs Kaimal model', fontsize=12)
ax.legend(fontsize=8, loc='lower left')
ax.grid(True, which='both', ls=':', alpha=0.4)

fig.tight_layout()
plt.show()

# Fit inertial subrange slope for the heat flux cospectrum
slope, r2, _ = fit_inertial_slope(res.freq_nd, res.ncosp_wT)
expected = -4.0 / 3.0
print(f'\nInertial subrange slope: {slope:.3f}  (expected: {expected:.3f},  R2 = {r2:.3f})')
print('A slope near -4/3 indicates a well-resolved inertial subrange.')

## 5. Built-in Spectral Plots

The `eccospectra.plotting` module provides three publication-ready figures:

| Function | What it shows |
|----------|---------------|
| `plot_cospectra()` | 2x2 grid of normalized cospectra ($w'T'$, $w'u'$, $w'CO_2'$, $w'H_2O'$) colored by $z/L$ |
| `plot_spectra()` | 2x2 grid of normalized power spectra ($u$, $v$, $w$, $T_s$) |
| `plot_ogive()` | 4-panel cumulative cospectra (convergence check for averaging period) |

All three accept a `stability_range` filter. With only one 30-minute interval in this example
we widen the range to ensure the interval is always included.

When processing many files the colors will span from blue (strongly unstable, $z/L \ll 0$)
through white (neutral) to red (strongly stable, $z/L \gg 0$).

In [ ]:
# 2x2 normalized cospectra with Kaimal model overlay and -4/3 reference slope
fig, axes = plot_cospectra(
    results,
    stability_range=(-100, 100),  # wide range to always include the single interval
    show_model=True,
    show_slope=True,
)
fig.suptitle('Normalized Cospectra -- Single 30-min Interval', y=1.01, fontsize=13)
plt.show()

In [ ]:
# 2x2 normalized power spectra (variance-normalized, -2/3 reference slope)
fig, axes = plot_spectra(results, stability_range=(-100, 100))
fig.suptitle('Normalized Power Spectra -- Single 30-min Interval', y=1.01, fontsize=13)
plt.show()

In [ ]:
# Ogives (cumulative cospectra from high to low frequency, normalized by covariance)
# Interpretation:
#   - Ogive converges to 1.0 at low f  => 30-min period captures all flux  (good)
#   - Ogive still rising at low f       => flux loss; need longer averaging period
fig, axes = plot_ogive(results, stability_range=(-100, 100))
fig.suptitle('Ogives -- Cumulative Cospectra (Averaging Period Convergence Check)', fontsize=12)
plt.show()

## 6. Spectral Transfer Functions

Real instruments do not measure all eddy frequencies equally well. Each physical process
introduces a **transfer function** $T(n) \in [0, 1]$ that describes the fraction of the
true signal captured at frequency $n$.

**Low-frequency loss** (large eddies attenuated):
- Block-average (finite averaging window) — Kaimal et al. (1968)
- Linear detrend — Rannik & Vesala (1999)

**High-frequency loss** (small eddies attenuated):
- First-order sensor response (time constant $\tau$) — Moore (1986)
- Sonic path averaging (line averaging over path length $l$) — Kaimal et al. (1968)
- Scalar (IRGA) path averaging — Moore (1986)
- Sensor separation between sonic and IRGA — Moore (1986)

The **combined** transfer function is the product of all applicable individual TFs.

In [ ]:
u_mean = results[0].u_mean
freq = np.logspace(-3, 1.5, 500)  # 0.001 -> ~32 Hz

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle(
    f'Individual Spectral Transfer Functions  (mean U = {u_mean:.1f} m/s)',
    fontsize=13
)

# (a) Block-average: low-frequency loss from finite averaging window
ax = axes[0, 0]
ax.semilogx(freq, tf_block_average(freq, 30.0), 'b-',  lw=2, label='T_avg = 30 min')
ax.semilogx(freq, tf_block_average(freq, 60.0), 'r--', lw=2, label='T_avg = 60 min')
ax.axvline(1 / 1800.0, color='grey', ls=':', lw=1.2, label='f = 1/(30 min)')
ax.set_xlabel('Frequency n  [Hz]')
ax.set_ylabel('T(n)')
ax.set_title('(a) Block-average (low-freq loss)')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
ax.grid(True, ls=':', alpha=0.4)

# (b) First-order sensor response: high-frequency rolloff
ax = axes[0, 1]
for tau, ls, lbl in [
    (0.0,  '-',  'tau = 0 s (ideal)'),
    (0.1,  '--', 'tau = 0.1 s  (IRGASON CO2/H2O)'),
    (0.5,  ':',  'tau = 0.5 s'),
    (1.0,  '-.', 'tau = 1.0 s'),
]:
    ax.semilogx(freq, tf_first_order_response(freq, tau), lw=2, ls=ls, label=lbl)
ax.set_xlabel('Frequency n  [Hz]')
ax.set_ylabel('T(n)')
ax.set_title('(b) First-order sensor response')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=8)
ax.grid(True, ls=':', alpha=0.4)

# (c) Path averaging: sonic (sinc^2) vs IRGA optical path
ax = axes[1, 0]
ax.semilogx(freq, tf_sonic_line_averaging(freq, u_mean, 0.05),  '-',  lw=2, label='Sonic  L = 5 cm')
ax.semilogx(freq, tf_sonic_line_averaging(freq, u_mean, 0.10),  '--', lw=2, label='Sonic  L = 10 cm  (IRGASON)')
ax.semilogx(freq, tf_sonic_line_averaging(freq, u_mean, 0.20),  ':',  lw=2, label='Sonic  L = 20 cm')
ax.semilogx(freq, tf_scalar_path_averaging(freq, u_mean, 0.154), '-.', lw=2, label='IRGA   L = 15.4 cm  (IRGASON)')
ax.set_xlabel('Frequency n  [Hz]')
ax.set_ylabel('T(n)')
ax.set_title('(c) Path averaging (sonic + IRGA)')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=8)
ax.grid(True, ls=':', alpha=0.4)

# (d) Sensor separation: lateral distance between sonic and IRGA
ax = axes[1, 1]
for sep, ls, lbl in [
    (0.0,  '-',  'd = 0 m  (integrated, IRGASON)'),
    (0.05, '--', 'd = 5 cm'),
    (0.15, ':',  'd = 15 cm'),
    (0.30, '-.', 'd = 30 cm'),
]:
    ax.semilogx(freq, tf_sensor_separation(freq, u_mean, sep), lw=2, ls=ls, label=lbl)
ax.set_xlabel('Frequency n  [Hz]')
ax.set_ylabel('T(n)')
ax.set_title('(d) Sensor separation')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=8)
ax.grid(True, ls=':', alpha=0.4)

fig.tight_layout()
plt.show()

In [ ]:
# Combined transfer function T(n) = product of all applicable individual TFs
# Different flux variables have different transfer functions because
# the IRGA introduces additional high-frequency loss for CO2 and H2O.

freq = np.logspace(-3, 1.5, 500)

colors_ft = {'wT': 'steelblue', 'wu': 'firebrick', 'wCO2': 'seagreen', 'wH2O': 'darkorchid'}
labels_ft = {
    'wT':   'Heat flux (wT)   -- sonic T only',
    'wu':   'Momentum (wu)    -- sonic only',
    'wCO2': 'CO2 flux (wCO2)  -- IRGA path + tau_co2',
    'wH2O': 'H2O flux (wH2O)  -- IRGA path + tau_h2o',
}

fig, ax = plt.subplots(figsize=(9, 5))

for ft in ['wT', 'wu', 'wCO2', 'wH2O']:
    T = combined_transfer_function(
        freq, u_mean, instrument,
        averaging_period=config.averaging_period,
        flux_type=ft,
    )
    ax.semilogx(freq, T, lw=2, color=colors_ft[ft], label=labels_ft[ft])

ax.axvline(1 / (config.averaging_period * 60), color='grey', ls='--', lw=1,
           label='Low-freq cutoff  f = 1/T_avg')
nyq = config.sampling_freq / 2
ax.axvline(nyq, color='grey', ls=':', lw=1, label=f'Nyquist  ({nyq:.0f} Hz)')

ax.set_xlabel(
    f'Frequency n  [Hz]   (mean U = {u_mean:.1f} m/s,  z_eff = {config.z_eff:.1f} m)',
    fontsize=11
)
ax.set_ylabel('Combined transfer function T(n)', fontsize=11)
ax.set_title(f'Combined Transfer Functions -- {instrument.model}', fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9, loc='lower left')
ax.grid(True, which='both', ls=':', alpha=0.4)

plt.tight_layout()
plt.show()

print('Note: the IRGASON integrated design gives near-ideal T(n) across the turbulent range.')
print('Separate sensors (e.g., sonic + LI-7500) would show more high-frequency attenuation.')

## 7. Spectral Correction Factors

The correction factor $CF$ is the ratio of the true flux to the measured (attenuated) flux:

$$CF = \frac{\int Co_{\mathrm{model}}(f)\,d(\ln f)}{\int T(f)\cdot Co_{\mathrm{model}}(f)\,d(\ln f)} \geq 1$$

where $T(f)$ is the combined transfer function and $Co_{\mathrm{model}}(f)$ is the Kaimal
(1972) model cospectrum used as a proxy for the true spectral shape.

The corrected flux is: $F_{\mathrm{corrected}} = CF \times F_{\mathrm{measured}}$

**Two implementations are available:**
- `method='massman'` — full numerical integration (Massman 2000); more accurate
- `method='horst'` — closed-form approximation (Horst 1997); faster, less accurate

CF increases with wind speed because faster winds shift turbulence to higher frequencies
where sensor attenuation is greater.

In [ ]:
# --- CF vs wind speed (Massman 2000) ---
u_speeds = np.linspace(0.5, 12.0, 40)
flux_types = ['wT', 'wu', 'wCO2', 'wH2O']
colors_ft  = {'wT': 'steelblue', 'wu': 'firebrick', 'wCO2': 'seagreen', 'wH2O': 'darkorchid'}
labels_ft  = {'wT': 'Heat (wT)', 'wu': 'Momentum (wu)', 'wCO2': 'CO2', 'wH2O': 'H2O'}

fig, ax = plt.subplots(figsize=(9, 5))
for ft in flux_types:
    cfs = [
        compute_spectral_correction_factor(
            u_mean=u, z_eff=config.z_eff, instrument=instrument,
            averaging_period=config.averaging_period, flux_type=ft,
        ) for u in u_speeds
    ]
    ax.plot(u_speeds, cfs, lw=2, color=colors_ft[ft], label=labels_ft[ft])

u_example = results[0].u_mean
ax.axvline(u_example, color='k', ls='--', lw=1.5, label=f'Example: U = {u_example:.1f} m/s')
ax.set_xlabel('Mean wind speed U  [m/s]', fontsize=12)
ax.set_ylabel('Correction factor CF', fontsize=12)
ax.set_title('Massman (2000) Spectral Correction Factor vs Wind Speed', fontsize=12)
ax.set_ylim(0.99, 1.4)
ax.legend(fontsize=10)
ax.grid(True, ls=':', alpha=0.4)
plt.tight_layout()
plt.show()

# --- Massman vs Horst comparison for the example interval ---
print(f'\nCorrection factors at U = {u_example:.2f} m/s, z_eff = {config.z_eff:.2f} m:')
print('{:>8}  {:>12}  {:>12}  {:>10}'.format('Flux', 'Massman CF', 'Horst CF', 'Diff %'))
print('-' * 48)
for ft in flux_types:
    cf_m = compute_spectral_correction_factor(
        u_example, config.z_eff, instrument, config.averaging_period, ft
    )
    # Effective time constant for Horst method (combine sensor + path)
    tau_s    = instrument.tau_T if ft == 'wT' else (
               0.0 if ft == 'wu' else (
               instrument.tau_co2 if ft == 'wCO2' else instrument.tau_h2o))
    tau_path = (instrument.irga_path_length / (2 * np.pi * u_example)
                if ft not in ('wT', 'wu') else 0.0)
    tau_eff  = (tau_s ** 2 + tau_path ** 2) ** 0.5
    cf_h     = horst_analytical_correction(u_example, config.z_eff, tau_eff, ft)
    diff     = 100.0 * (cf_h - cf_m) / cf_m if cf_m > 0 else float('nan')
    print(f'{ft:>8}  {cf_m:>12.5f}  {cf_h:>12.5f}  {diff:>10.3f}')

## 8. Apply Spectral Corrections

`apply_spectral_corrections()` applies corrections in-place to the `SpectralResult` objects.
It:
1. Computes $CF$ for each flux type using the chosen method
2. Scales the covariance: `cov_corrected = CF * cov_raw`
3. Divides the cospectral array by the transfer function $T(f)$ at each frequency bin
4. Optionally applies the WPL density correction (for open-path IRGAs)

Corrected covariances are stored in `res.qc_flags['cov_wT_corrected']`, etc.  
Correction factors are stored in `res.qc_flags['cf_wT']`, etc.

> **Note:** Call `enrich_results_with_means()` first to add mean CO$_2$, H$_2$O,
> and pressure values needed for the WPL correction.

In [ ]:
# Add mean scalar concentrations (needed for WPL) to SpectralResult objects
results = enrich_results_with_means(results, df_clean, config)

# Deep-copy so we can compare corrected vs. uncorrected side by side
results_uncorr = copy.deepcopy(results)
results_corr   = copy.deepcopy(results)

# Apply high-frequency + low-frequency corrections (Massman 2000) and WPL
results_corr = apply_spectral_corrections(
    results_corr,
    site_config=config,
    instrument=instrument,
    apply_high_freq=True,
    apply_low_freq=True,
    apply_wpl=True,
    method='massman',
    verbose=True,
)

In [ ]:
# --- Compare corrected vs uncorrected cospectra ---
res_u = results_uncorr[0]
res_c = results_corr[0]

mask = np.isfinite(res_u.freq_nd) & (res_u.freq_nd > 0)
f_nd = res_u.freq_nd[mask]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Spectral Corrections: Before vs After (Massman 2000)', fontsize=13)

panels = [
    (axes[0, 0], 'ncosp_wT',   'Heat flux  nCo(wT) / cov(wT)',     'wT'),
    (axes[0, 1], 'ncosp_wu',   'Momentum  nCo(wu) / cov(wu)',       'wu'),
    (axes[1, 0], 'ncosp_wCO2', 'CO2 flux  nCo(wCO2) / cov(wCO2)',  'wCO2'),
    (axes[1, 1], 'ncosp_wH2O', 'H2O flux  nCo(wH2O) / cov(wH2O)',  'wH2O'),
]

for ax, attr, title, ft in panels:
    co_u = np.abs(getattr(res_u, attr))[mask]
    co_c = np.abs(getattr(res_c, attr))[mask]

    pos_u = co_u > 1e-6
    pos_c = co_c > 1e-6
    if pos_u.any():
        ax.loglog(f_nd[pos_u], co_u[pos_u], 'b-', lw=1.5, alpha=0.9, label='Uncorrected')
    if pos_c.any():
        ax.loglog(f_nd[pos_c], co_c[pos_c], 'r-', lw=1.5, alpha=0.9, label='Corrected')

    f_m = np.logspace(-3, 2, 500)
    ax.loglog(f_m, _kaimal_cospec_model(f_m, ft), 'k-', lw=2, alpha=0.5, label='Kaimal (1972)')

    cf_key = f'cf_{ft}'
    cf_val = res_c.qc_flags.get(cf_key, float('nan'))
    ax.set_title(f'{title}   (CF = {cf_val:.4f})', fontsize=10)
    ax.set_xlabel('f = nz/U', fontsize=10)
    ax.set_ylabel('n * Co(n) / cov(wx)', fontsize=10)
    ax.set_xlim(1e-2, 20)
    ax.set_ylim(1e-3, 5)
    ax.grid(True, which='both', ls=':', alpha=0.4)
    ax.legend(fontsize=9)

fig.tight_layout()
plt.show()

# Corrections are most visible at high f where T(n) < 1.
# The corrected cospectrum should lie closer to the Kaimal model.

## 9. WPL Density Correction

Open-path gas analysers measure **number density** (or mass density). Vertical transport
of sensible heat and water vapour causes air density fluctuations that are not actual
CO$_2$ or H$_2$O flux. The **Webb-Pearman-Leuning (WPL 1980)** correction accounts for this:

$$F_{c,\mathrm{wpl}} = \bar{F}_c + \mu\,\frac{\bar{\rho}_c}{\bar{\rho}_d}\,\bar{F}_e
   + (1 + \mu\sigma)\,\frac{\bar{\rho}_c}{\bar{T}}\,\frac{\bar{H}}{\bar{\rho}_d\,c_p}$$

where $\mu = M_d/M_v \approx 1.608$ and $\sigma = \bar{\rho}_v/\bar{\rho}_d$.

`apply_spectral_corrections()` automatically applies WPL when `instrument.irga_type == 'open_path'`
and mean scalar concentrations are available. The section below demonstrates the manual call
to `wpl_correction()` to inspect the correction terms.

In [ ]:
res = results_corr[0]

# Mean scalar densities added by enrich_results_with_means()
co2_mean = getattr(res, 'co2_mean', None)
h2o_mean = getattr(res, 'h2o_mean', None)
P_mean   = getattr(res, 'P_mean', 101.3)  # kPa; uses std. atm. if PA not in TOA5 file

# Fallback if enrich_results_with_means found no matching columns
if co2_mean is None:
    co2_mean = float(df_raw['CO2_density'].mean())
if h2o_mean is None:
    h2o_mean = float(df_raw['H2O_density'].mean())

print('Mean scalar concentrations (from 30-min interval):')
print(f'  CO2:  {co2_mean:.2f} mg/m3')
print(f'  H2O:  {h2o_mean:.3f} g/m3')
print(f'  P:    {P_mean:.1f} kPa  (standard atmosphere)')
print(f'  T:    {res.T_mean:.2f} degC')

# Use spectrally-corrected covariances as inputs to WPL
Fc = res.qc_flags.get('cov_wCO2_corrected', res.cov_wCO2)  # mg/m2/s
Fe = res.qc_flags.get('cov_wH2O_corrected', res.cov_wH2O)  # g/m2/s
H  = res.qc_flags.get('cov_wT_corrected',   res.cov_wT) * 1200.0  # W/m2

print('\n--- Spectrally corrected (pre-WPL) ---')
print(f'  Fc (CO2 flux):  {Fc:.6f} mg/m2/s')
print(f'  Fe (H2O flux):  {Fe:.6f} g/m2/s')
print(f'  H  (heat flux): {H:.3f} W/m2')

# Manual WPL correction
wpl = wpl_correction(
    Fc_raw=Fc,
    Fe_raw=Fe,
    H=H,
    T_mean=res.T_mean,
    P_mean=P_mean,
    co2_mean=co2_mean,
    h2o_mean=h2o_mean,
)

fc_wpl  = wpl['Fc_wpl']
fe_wpl  = wpl['Fe_wpl']
dfc     = wpl['Fc_correction']
dfe     = wpl['Fe_correction']
mu_val  = wpl['mu']
sig_val = wpl['sigma']

print('\n--- After WPL correction ---')
print(f'  Fc_wpl:        {fc_wpl:.6f} mg/m2/s')
print(f'  Fe_wpl:        {fe_wpl:.6f} g/m2/s')
print(f'  WPL term DFc:  {dfc:.6f} mg/m2/s  ({100 * dfc / Fc:.2f}% of Fc)' if abs(Fc) > 1e-9 else f'  WPL term DFc:  {dfc:.6f} mg/m2/s')
print(f'  WPL term DFe:  {dfe:.6f} g/m2/s   ({100 * dfe / Fe:.2f}% of Fe)' if abs(Fe) > 1e-9 else f'  WPL term DFe:  {dfe:.6f} g/m2/s')
print(f'  mu = Md/Mv:    {mu_val:.5f}')
print(f'  sigma = pv/pd: {sig_val:.7f}')

## 10. Summary

A complete correction chain typically increases measured fluxes by 5-20% depending on
wind speed, instrument configuration, and stability. The table below shows the net
effect for the example interval.

In [ ]:
res_u = results_uncorr[0]
res_c = results_corr[0]

print('=' * 72)
print('SPECTRAL CORRECTION SUMMARY')
print('=' * 72)
print(f'  Site:        z = {config.z_measurement} m, z_canopy = {config.z_canopy} m,'
      f'  z_eff = {config.z_eff:.2f} m')
print(f'  Instrument:  {instrument.model}  ({instrument.irga_type})')
print(f'  Mean U:      {res_c.u_mean:.2f} m/s')
print(f'  z/L:         {res_c.zL:.4f}')
print()

# Flux comparison table
header = '{:>28}  {:>14}  {:>14}  {:>9}'.format(
    'Quantity', 'Uncorrected', 'Corrected', 'Change'
)
print(header)
print('-' * 72)

H_corr = res_c.qc_flags.get('cov_wT_corrected', float('nan')) * 1200.0
flux_map = [
    ('H  [W/m2]',            res_u.H,         H_corr),
    ('cov(wu)  [m2/s2]',     res_u.cov_wu,    res_c.qc_flags.get('cov_wu_corrected',   float('nan'))),
    ('cov(wCO2)  [mg/m2/s]', res_u.cov_wCO2,  res_c.qc_flags.get('cov_wCO2_corrected', float('nan'))),
    ('cov(wH2O)  [g/m2/s]',  res_u.cov_wH2O,  res_c.qc_flags.get('cov_wH2O_corrected', float('nan'))),
]
for lbl, val_u, val_c in flux_map:
    if np.isfinite(val_u) and np.isfinite(val_c) and abs(val_u) > 1e-10:
        pct = 100.0 * (val_c - val_u) / abs(val_u)
        print(f'{lbl:>28}  {val_u:>14.5f}  {val_c:>14.5f}  {pct:>8.2f}%')
    else:
        print(f'{lbl:>28}  {val_u:>14.5f}  {"N/A":>14}')

print()
print('Correction factors (Massman 2000):')
for ft in ['wT', 'wu', 'wCO2', 'wH2O']:
    cf = res_c.qc_flags.get(f'cf_{ft}', float('nan'))
    if np.isfinite(cf):
        print(f'  CF({ft}): {cf:.5f}   (+{(cf - 1.0) * 100:.3f}% increase)')

if 'wpl_Fc' in res_c.qc_flags:
    wpl_fc = res_c.qc_flags['wpl_Fc']
    wpl_fe = res_c.qc_flags['wpl_Fe']
    print()
    print('WPL-corrected fluxes (stored by apply_spectral_corrections):')
    print(f'  Fc_wpl = {wpl_fc:.6f} mg/m2/s')
    print(f'  Fe_wpl = {wpl_fe:.6f} g/m2/s')

## References

- **Kaimal, J.C. et al. (1972).** Spectral characteristics of surface-layer turbulence.
  *Quart. J. Roy. Meteor. Soc.*, 98, 563-589.
- **Massman, W.J. (2000).** A simple method for estimating frequency response corrections
  for eddy covariance systems. *Agric. For. Meteorol.*, 104, 185-198.
- **Horst, T.W. (1997).** A simple formula for attenuation of eddy fluxes measured with
  first-order-response scalar sensors. *Boundary-Layer Meteorol.*, 82, 219-233.
- **Moore, C.J. (1986).** Frequency response corrections for eddy correlation systems.
  *Boundary-Layer Meteorol.*, 37, 17-35.
- **Webb, E.K., Pearman, G.I. & Leuning, R. (1980).** Correction of flux measurements
  for density effects due to heat and water vapour transfer.
  *Quart. J. Roy. Meteor. Soc.*, 106, 85-100.
- **Metzger, S. et al. (2012).** Eddy-covariance flux measurements with a weight-shift
  microlight aircraft. *Atmos. Meas. Tech.*, 5, 1699-1717.